In [ ]:
#Install needed libraries and dependencies
!pip uninstall -y vllm transformers tokenizers
!pip install vllm==0.6.4.post1 transformers==4.46.3 tokenizers==0.20.3 accelerate pandas

Llama-3.2-1B-Instruct

In [ ]:
#Tier 3 Voting
import os
import gc
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from google.colab import drive

gc.collect()
torch.cuda.empty_cache()
drive.mount('/content/drive')

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MASTER_CSV_PATH = "PATH_OF_TIER_2_OUTPUT_RATIONALES"
OUTPUT_CSV_PATH = "PATH_OF_OUTPUT"

print("Loading Tokenizer and vLLM Engine...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm = LLM(
    model=MODEL_NAME,
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    enforce_eager=True
)

print(f"Loading Master Rationale Data...")
df = pd.read_csv(MASTER_CSV_PATH)

target_map = {'option1': 'A', 'option2': 'B', 'option3': 'C', 'option4': 'D'}
df['True_Label'] = df['target'].map(target_map)

print("Formatting prompts for Logical, Balanced, and Creative instances...")
all_prompts = []

#Order: [Row0_Logical, Row0_Balanced, Row0_Creative, Row1_Logical]
for _, row in df.iterrows():
    options_block = f"A. {row['option1']}\nB. {row['option2']}\nC. {row['option3']}\nD. {row['option4']}"

    rationales = [
        row.get('rationale_Logical', "No rationale provided."),
        row.get('rationale_Balanced', "No rationale provided."),
        row.get('rationale_Creative', "No rationale provided.")
    ]
#Prompt
    for rationale in rationales:
        messages = [
            {"role": "system", "content": "You are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
            {"role": "user", "content": f"### RATIONALE\n{rationale}\n\n### QUESTION\n{row['question']}\n\n### OPTIONS\n{options_block}\n\nThe correct option is:"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        all_prompts.append(text)

params = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    logprobs=20
)

print(f"\nRunning vLLM Log-Likelihood Extraction for {len(all_prompts)} total prompts...")
outputs = llm.generate(all_prompts, params)

print("\nCalculating Majority Votes and Accuracies...")
results_log = []
correct_count = 0
valid_choices = ['A', 'B', 'C', 'D']

def get_best_letter(logprobs_dict):
    best_letter = "C" #Fallback
    highest_prob = -float('inf')

#LogProb
    for token_id, logprob_obj in logprobs_dict.items():
        token_str = logprob_obj.decoded_token.strip().upper()
        if token_str in valid_choices and logprob_obj.logprob > highest_prob:
            highest_prob = logprob_obj.logprob
            best_letter = token_str

    return best_letter

for i in range(len(df)):
    row = df.iloc[i]
    truth = row['True_Label']

    out_logical = outputs[i * 3]
    out_balanced = outputs[i * 3 + 1]
    out_creative = outputs[i * 3 + 2]

    pred_logical = get_best_letter(out_logical.outputs[0].logprobs[0])
    pred_balanced = get_best_letter(out_balanced.outputs[0].logprobs[0])
    pred_creative = get_best_letter(out_creative.outputs[0].logprobs[0])

    preds = [pred_logical, pred_balanced, pred_creative]

    counts = Counter(preds)
    max_count = max(counts.values())
    tied_options = [opt for opt, count in counts.items() if count == max_count]

#Tiebreaker
    if len(tied_options) == 1:
        final_pred = tied_options[0]
    else:
        final_pred = pred_logical

    is_correct = (final_pred == truth)
    if is_correct:
        correct_count += 1

    results_log.append({
        "Question_ID": i,
        "True_Label": truth,
        "Pred_Logical": pred_logical,
        "Pred_Balanced": pred_balanced,
        "Pred_Creative": pred_creative,
        "Final_Voted_Pred": final_pred,
        "Is_Correct": is_correct
    })

accuracy = (correct_count / len(df)) * 100
print(f"\nENSEMBLE VOTING COMPLETE!")
print(f"FINAL ACCURACY: {accuracy:.2f}%")

final_df = pd.DataFrame(results_log)
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Voting breakdown saved to:\n{OUTPUT_CSV_PATH}")

Gemma-2-2B-Instruct

In [ ]:
#Tier 3 Voting
import os
import gc
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

if 'llm' in globals():
    del llm
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "google/gemma-2-2b-it"
MASTER_CSV_PATH = "PATH_OF_TIER_2_OUTPUT_RATIONALES"
OUTPUT_CSV_PATH = "PATH_OF_OUTPUT"

print(f"Loading {MODEL_NAME} via Hugging Face Transformers...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

choices = ["A", "B", "C", "D"]
choice_tokens = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]

print(f"Loading Master Rationale Data...")
df = pd.read_csv(MASTER_CSV_PATH)
target_map = {'option1': 'A', 'option2': 'B', 'option3': 'C', 'option4': 'D'}
df['True_Label'] = df['target'].map(target_map)

results_log = []
correct_count = 0

print("Running Native HF Log-Likelihood Extraction...")
for i in tqdm(range(len(df)), desc="Evaluating Gemma-2"):
    row = df.iloc[i]
    truth = row['True_Label']
    options_block = f"A. {row['option1']}\nB. {row['option2']}\nC. {row['option3']}\nD. {row['option4']}"

    rationales = [
        row.get('rationale_Logical', "No rationale."),
        row.get('rationale_Balanced', "No rationale."),
        row.get('rationale_Creative', "No rationale.")
    ]

    preds = []

#Prompt
    for rationale in rationales:
        messages = [
            {"role": "user", "content": f"You are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D).\n\n### RATIONALE\n{rationale}\n\n### QUESTION\n{row['question']}\n\n### OPTIONS\n{options_block}\n\nThe correct option is:"}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#LogProb
        with torch.no_grad():
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]

        scores = {c: next_token_logits[tok].item() for c, tok in zip(choices, choice_tokens)}
        best_letter = max(scores, key=scores.get)
        preds.append(best_letter)

    pred_logical, pred_balanced, pred_creative = preds

    counts = Counter(preds)
    max_count = max(counts.values())
    tied_options = [opt for opt, count in counts.items() if count == max_count]

#Tiebreaker
    final_pred = tied_options[0] if len(tied_options) == 1 else pred_logical

    is_correct = (final_pred == truth)
    if is_correct:
        correct_count += 1

    results_log.append({
        "Question_ID": i,
        "True_Label": truth,
        "Pred_Logical": pred_logical,
        "Pred_Balanced": pred_balanced,
        "Pred_Creative": pred_creative,
        "Final_Voted_Pred": final_pred,
        "Is_Correct": is_correct
    })

accuracy = (correct_count / len(df)) * 100
print(f"\nGEMMA-2 ENSEMBLE VOTING COMPLETE!")
print(f"FINAL ACCURACY: {accuracy:.2f}%")

final_df = pd.DataFrame(results_log)
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Voting breakdown saved to:\n{OUTPUT_CSV_PATH}")

In [ ]:
#Tier 3 Voting
import os
import gc
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from google.colab import drive

os.environ.pop("VLLM_ATTENTION_BACKEND", None)

if 'llm' in globals():
    del llm
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
drive.mount('/content/drive')

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
MASTER_CSV_PATH = "PATH_OF_TIER_2_OUTPUT_RATIONALES"
OUTPUT_CSV_PATH = "PATH_OF_OUTPUT"

print(f"Loading {MODEL_NAME} into vLLM...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm = LLM(
    model=MODEL_NAME,
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    enforce_eager=True
)

print("Loading Master Rationale Data...")
df = pd.read_csv(MASTER_CSV_PATH)

target_map = {'option1': 'A', 'option2': 'B', 'option3': 'C', 'option4': 'D'}
df['True_Label'] = df['target'].map(target_map)

print("Formatting prompts for Logical, Balanced, and Creative instances...")
all_prompts = []

for _, row in df.iterrows():
    options_block = f"A. {row['option1']}\nB. {row['option2']}\nC. {row['option3']}\nD. {row['option4']}"

    rationales = [
        row.get('rationale_Logical', "No rationale provided."),
        row.get('rationale_Balanced', "No rationale provided."),
        row.get('rationale_Creative', "No rationale provided.")
    ]

#Prompt
    for rationale in rationales:
        messages = [
            {"role": "system", "content": "You are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
            {"role": "user", "content": f"### RATIONALE\n{rationale}\n\n### QUESTION\n{row['question']}\n\n### OPTIONS\n{options_block}\n\nThe correct option is:"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        all_prompts.append(text)

params = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    logprobs=20
)

print(f"\nRunning vLLM Log-Likelihood Extraction for {len(all_prompts)} total prompts...")
outputs = llm.generate(all_prompts, params)

print("\nCalculating Majority Votes and Accuracies...")
results_log = []
correct_count = 0
valid_choices = ['A', 'B', 'C', 'D']

def get_best_letter(logprobs_dict):
    best_letter = "C" # Default fallback
    highest_prob = -float('inf')

#LogProb
    for token_id, logprob_obj in logprobs_dict.items():
        token_str = logprob_obj.decoded_token.strip().upper()
        if token_str in valid_choices and logprob_obj.logprob > highest_prob:
            highest_prob = logprob_obj.logprob
            best_letter = token_str

    return best_letter

for i in range(len(df)):
    row = df.iloc[i]
    truth = row['True_Label']

    out_logical = outputs[i * 3]
    out_balanced = outputs[i * 3 + 1]
    out_creative = outputs[i * 3 + 2]

    pred_logical = get_best_letter(out_logical.outputs[0].logprobs[0])
    pred_balanced = get_best_letter(out_balanced.outputs[0].logprobs[0])
    pred_creative = get_best_letter(out_creative.outputs[0].logprobs[0])

    preds = [pred_logical, pred_balanced, pred_creative]

#Tiebreaker
    counts = Counter(preds)
    max_count = max(counts.values())
    tied_options = [opt for opt, count in counts.items() if count == max_count]

    if len(tied_options) == 1:
        final_pred = tied_options[0]
    else:
        final_pred = pred_logical

    is_correct = (final_pred == truth)
    if is_correct:
        correct_count += 1

    results_log.append({
        "Question_ID": i,
        "True_Label": truth,
        "Pred_Logical": pred_logical,
        "Pred_Balanced": pred_balanced,
        "Pred_Creative": pred_creative,
        "Final_Voted_Pred": final_pred,
        "Is_Correct": is_correct
    })

accuracy = (correct_count / len(df)) * 100
print(f"\nLLAMA-3.2-3B ENSEMBLE VOTING COMPLETE!")
print(f"FINAL ACCURACY: {accuracy:.2f}%")

final_df = pd.DataFrame(results_log)
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Voting breakdown saved to:\n{OUTPUT_CSV_PATH}")